#FDIC Community Banks Python API Requests 

In [ ]:
##Pull data from FDIC API for all active banks in California and save to CSV

import requests
import pandas as pd

url = "https://banks.data.fdic.gov/api/institutions"

params = {
    "filters": "STALP:CA AND ACTIVE:1",
    "fields": "CERT,NAME,CITY,STALP,ESTYMD,BKCLASS",
    "limit": 10000,
    "format": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()
institutions = pd.json_normalize([record["data"] for record in data["data"]])

print(f"Pulled {len(institutions)} California institutions")
print(institutions.head())

institutions.to_csv("ca_institutions.csv", index=False)


Pulled 116 California institutions
           CITY  ACTIVE BKCLASS   CERT STALP      ESTYMD  \
0    Long Beach       1      SM   1225    CA  11/20/1907   
1          Lodi       1      NM   1331    CA  05/24/1916   
2      Stockton       1      NM   1536    CA  08/12/1867   
3   Los Angeles       1       N  17281    CA  12/04/1953   
4  Walnut Creek       1      NM   1768    CA  01/01/1905   

                                             NAME     ID  
0        Farmers and Merchants Bank of Long Beach   1225  
1  Farmers & Merchants Bank of Central California   1331  
2                                Bank of Stockton   1536  
3                              City National Bank  17281  
4                                  Mechanics Bank   1768  


In [27]:
ca_institutions = pd.read_csv("ca_institutions.csv")

for col in ca_institutions.select_dtypes(include="object").columns:
   ca_institutions[col] = ca_institutions[col].astype(str).str.replace('"', "'", regex=False)


ca_institutions.to_csv("ca_institutions_reformatted.csv", index=False)

/var/folders/4g/7t5q5qys145_ll4nj4bl8dwh0000gn/T/ipykernel_19304/2370373728.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in ca_institutions.select_dtypes(include="object").columns:


In [ ]:
## Pull financial data for all active banks in California and save to CSV

import time

institutions = pd.read_csv("ca_institutions.csv")
certs = institutions["CERT"].astype(str).tolist()

url = "https://banks.data.fdic.gov/api/financials"
fields = "CERT,REPDTE,ASSET,DEP,NETINC,ROA,ROE,EEFFR,LNRE,LNCI,LNCON,NUMEMP"

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

all_financials = []

for batch in chunk_list(certs, 50):
    cert_filter = " OR ".join([f"CERT:{c}" for c in batch])
    filters = f"({cert_filter}) AND REPDTE:[20220101 TO 20251231]"

    params = {
        "filters": filters,
        "fields": fields,
        "limit": 10000,
        "format": "json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    records = [r["data"] for r in data["data"]]
    all_financials.extend(records)

    time.sleep(0.2)

financials = pd.json_normalize(all_financials)

print(f"Pulled {len(financials)} bank-quarter records")
print(financials.head())

financials.to_csv("ca_financials_2022_2025.csv", index=False)


Pulled 1830 bank-quarter records
      LNCON       ROA     LNRE    REPDTE  NUMEMP   ROE     ASSET  CERT  \
0  144095.0  0.963497  4828847  20220331   761.0  8.90  11673393  1225   
1  127342.0  0.980220  5160767  20220630   758.0  9.04  11731961  1225   
2  132202.0  0.952577  5754159  20220930   771.0  8.79  11937528  1225   
3  151316.0  0.926322  6070273  20221231   783.0  8.54  12054744  1225   
4  161182.0  0.718514  6118432  20230331   786.0  6.60  12021615  1225   

     LNCI    NETINC      EEFFR      DEP             ID  
0  117166   27835.0  55.727349  9359167  1225_20220331  
1  100512   56924.0  55.960073  9461867  1225_20220630  
2   99552   83557.0  55.688025  9350260  1225_20220930  
3  125087  109002.0  57.188749  9142496  1225_20221231  
4  138780   21624.0  64.268770  8969147  1225_20230331  


In [28]:
ca_financials = pd.read_csv("ca_financials_2022_2025.csv")

for col in ca_financials.select_dtypes(include="object").columns:
   ca_financials[col] = ca_financials[col].astype(str).str.replace('"', "'", regex=False)


ca_financials.to_csv("ca_financials_reformatted.csv", index=False)

/var/folders/4g/7t5q5qys145_ll4nj4bl8dwh0000gn/T/ipykernel_19304/2426604438.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in ca_financials.select_dtypes(include="object").columns:


In [ ]:
### Pull California GDP data from BEA API and save to CSV

import os

api_key = api_key = os.environ.get("BEA_API_KEY")
url = "https://apps.bea.gov/api/data"

params = {
    "UserID": api_key,
    "method": "GetData",
    "datasetname": "Regional",
    "TableName": "SQGDP1",
    "LineCode": 1,
    "GeoFips": "06000",
    "Year": "2022,2023,2024,2025",
    "ResultFormat": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

records = data["BEAAPI"]["Results"]["Data"]
state_gdp = pd.DataFrame(records)

print(f"Pulled {len(state_gdp)} rows")
print(state_gdp.head())

state_gdp.to_csv("ca_state_gdp_2022_2025.csv", index=False)


Pulled 16 rows
       Code GeoFips     GeoName TimePeriod                           CL_UNIT  \
0  SQGDP1-1   06000  California     2022Q2  Millions of chained 2017 dollars   
1  SQGDP1-1   06000  California     2023Q2  Millions of chained 2017 dollars   
2  SQGDP1-1   06000  California     2024Q2  Millions of chained 2017 dollars   
3  SQGDP1-1   06000  California     2025Q2  Millions of chained 2017 dollars   
4  SQGDP1-1   06000  California     2022Q1  Millions of chained 2017 dollars   

  UNIT_MULT  DataValue NoteRef  
0         6  3146315.9       1  
1         6  3191507.9       1  
2         6  3287040.8       1  
3         6  3377263.9       1  
4         6  3162422.4       1  


In [31]:
ca_state_gdp = pd.read_csv("ca_state_gdp_2022_2025.csv")

ca_state_gdp["STALP"] = "CA"

for col in ca_state_gdp.select_dtypes(include="object").columns:
   ca_state_gdp[col] = ca_state_gdp[col].astype(str).str.replace('"', "'", regex=False)


ca_state_gdp.to_csv("ca_state_gdp_reformatted.csv", index=False)

/var/folders/4g/7t5q5qys145_ll4nj4bl8dwh0000gn/T/ipykernel_19304/3538000557.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in ca_state_gdp.select_dtypes(include="object").columns:
